# Homework 4 BAN 5600
# Name: Ryan Cooper
# Course: BAN 5600
# Professor: Omar El Mghari
# Date: 4/19/2026

# Homework 4 - Problem 8
## Monitoring Shipment Delays and Missing Order Information

This notebook reviews shipment records, flags delayed deliveries and missing documents,
and creates an exception report for management review.

In [ ]:
# @title
import pandas as pd

# Sample shipment data
shipments = [
    {"Order_ID": 1001, "Customer_Name": "Alpha Stores", "Ship_Date": "2026-04-01",
     "Expected_Delivery": "2026-04-05", "Actual_Delivery": "2026-04-07",
     "Shipment_Status": "Delivered", "Missing_Documents": "No"},

    {"Order_ID": 1002, "Customer_Name": "Beta Supply", "Ship_Date": "2026-04-02",
     "Expected_Delivery": "2026-04-06", "Actual_Delivery": "",
     "Shipment_Status": "In Transit", "Missing_Documents": "Yes"},

    {"Order_ID": 1003, "Customer_Name": "Central Parts", "Ship_Date": "2026-04-03",
     "Expected_Delivery": "2026-04-07", "Actual_Delivery": "2026-04-07",
     "Shipment_Status": "Delivered", "Missing_Documents": "No"},

    {"Order_ID": 1004, "Customer_Name": "Delta Wholesale", "Ship_Date": "2026-04-04",
     "Expected_Delivery": "2026-04-08", "Actual_Delivery": "2026-04-11",
     "Shipment_Status": "Delayed", "Missing_Documents": "No"},

    {"Order_ID": 1005, "Customer_Name": "Empire Goods", "Ship_Date": "2026-04-05",
     "Expected_Delivery": "2026-04-09", "Actual_Delivery": "",
     "Shipment_Status": "In Transit", "Missing_Documents": "No"},

    {"Order_ID": 1006, "Customer_Name": "Fast Retail", "Ship_Date": "2026-04-06",
     "Expected_Delivery": "2026-04-10", "Actual_Delivery": "2026-04-13",
     "Shipment_Status": "Delivered", "Missing_Documents": "Yes"}
]

# Convert list of dictionaries to a DataFrame
df = pd.DataFrame(shipments)

# Convert date columns to datetime format
df["Ship_Date"] = pd.to_datetime(df["Ship_Date"])
df["Expected_Delivery"] = pd.to_datetime(df["Expected_Delivery"])
df["Actual_Delivery"] = pd.to_datetime(df["Actual_Delivery"], errors="coerce")

# Lists to store flagged orders
delayed_orders = []
missing_doc_orders = []
exception_rows = []

# Review each shipment record
for i in range(len(df)):
    row = df.loc[i]

    # Rule 1: Missing documents
    if row["Missing_Documents"] == "Yes":
        missing_doc_orders.append(row["Order_ID"])

    # Rule 2: Delivered late
    if pd.notna(row["Actual_Delivery"]) and row["Actual_Delivery"] > row["Expected_Delivery"]:
        delayed_orders.append(row["Order_ID"])

    # Rule 3: In transit but already past expected delivery
    if row["Shipment_Status"] == "In Transit" and pd.isna(row["Actual_Delivery"]):
        today_check = pd.Timestamp("2026-04-12")
        if today_check <= row["Expected_Delivery"]:
            continue
        delayed_orders.append(row["Order_ID"])

    # Add to exception report if any issue exists
    if row["Order_ID"] in delayed_orders or row["Order_ID"] in missing_doc_orders:
        exception_rows.append(row)

# Convert exception rows back to DataFrame
exception_report = pd.DataFrame(exception_rows)

print("===== Shipment Exception Summary =====")
print("Delayed shipments:", len(delayed_orders))
print("Shipments with missing documents:", len(missing_doc_orders))
print()

# Clean duplicate IDs using list logic
delayed_orders = list(dict.fromkeys(delayed_orders))
missing_doc_orders = list(dict.fromkeys(missing_doc_orders))

print("Delayed Order IDs:", delayed_orders)
print("Missing Document Order IDs:", missing_doc_orders)
print()

print("===== Final Exception Report =====")
print(exception_report)
print()

# While loop example: review urgent exceptions one by one
print("===== Urgent Review Preview =====")
index_position = 0
reviewed_count = 0

while index_position < len(exception_report):
    current_order = exception_report.iloc[index_position]

    if current_order["Missing_Documents"] == "No" and current_order["Shipment_Status"] == "Delivered":
        index_position += 1
        continue

    print("Reviewing Order:", current_order["Order_ID"], "-", current_order["Customer_Name"])
    reviewed_count += 1
    index_position += 1

    # Stop after reviewing first 3 urgent records
    if reviewed_count == 3:
        break

print()
print("Business Note:")
print("This automation helps logistics teams quickly identify delayed shipments")
print("and incomplete shipment records so they can follow up faster.")

# Homework 4 - Problem 2
## Tracking Low Inventory and Reorder Needs

This notebook reviews inventory records, flags low-stock items, and creates
a reorder report that can help support inventory control decisions.

In [ ]:
import pandas as pd

# Sample inventory data
inventory = [
    {"Item_ID": "A101", "Item_Name": "Printer Paper", "Category": "Office Supplies",
     "Current_Stock": 12, "Reorder_Level": 20, "Supplier": "Staples Co"},

    {"Item_ID": "A102", "Item_Name": "Blue Pens", "Category": "Office Supplies",
     "Current_Stock": 45, "Reorder_Level": 25, "Supplier": "Staples Co"},

    {"Item_ID": "B201", "Item_Name": "Coffee Pods", "Category": "Beverage",
     "Current_Stock": 8, "Reorder_Level": 15, "Supplier": "Coffee World"},

    {"Item_ID": "B202", "Item_Name": "Water Bottles", "Category": "Beverage",
     "Current_Stock": 30, "Reorder_Level": 20, "Supplier": "Aqua Supply"},

    {"Item_ID": "C301", "Item_Name": "Shipping Labels", "Category": "Warehouse",
     "Current_Stock": 4, "Reorder_Level": 10, "Supplier": "PackPro"},

    {"Item_ID": "C302", "Item_Name": "Packing Tape", "Category": "Warehouse",
     "Current_Stock": 18, "Reorder_Level": 18, "Supplier": "PackPro"}
]

# Convert list of dictionaries to DataFrame
df = pd.DataFrame(inventory)

# Lists to store low-stock and urgent items
low_stock_items = []
urgent_items = []
reorder_rows = []

# Review each inventory item
for i in range(len(df)):
    row = df.loc[i]

    # Skip items that are safely above the reorder level
    if row["Current_Stock"] > row["Reorder_Level"]:
        continue

    # Add item to low-stock list
    low_stock_items.append(row["Item_ID"])

    # Calculate shortage amount
    shortage = row["Reorder_Level"] - row["Current_Stock"]

    # Urgent rule: shortage of 5 or more units
    if shortage >= 5:
        urgent_items.append(row["Item_ID"])

    # Save row for reorder report
    reorder_rows.append({
        "Item_ID": row["Item_ID"],
        "Item_Name": row["Item_Name"],
        "Category": row["Category"],
        "Current_Stock": row["Current_Stock"],
        "Reorder_Level": row["Reorder_Level"],
        "Shortage": shortage,
        "Supplier": row["Supplier"]
    })

# Convert reorder rows to DataFrame
reorder_report = pd.DataFrame(reorder_rows)

# Sort reorder report from most urgent to least urgent
reorder_report = reorder_report.sort_values(by="Shortage", ascending=False)

print("===== Inventory Summary =====")
print("Total low-stock items:", len(low_stock_items))
print("Urgent low-stock items:", len(urgent_items))
print("Low-stock Item IDs:", low_stock_items)
print("Urgent Item IDs:", urgent_items)
print()

print("===== Reorder Report =====")
print(reorder_report)
print()

# While loop example: review urgent items one by one
print("===== Urgent Reorder Preview =====")
position = 0
reviewed = 0

while position < len(reorder_report):
    current_item = reorder_report.iloc[position]

    # If shortage is small, skip this row
    if current_item["Shortage"] < 5:
        position += 1
        continue

    print("Review Item:", current_item["Item_ID"], "-", current_item["Item_Name"])
    print("Supplier:", current_item["Supplier"])
    print("Shortage:", current_item["Shortage"])
    print()

    reviewed += 1
    position += 1

    # Stop after reviewing the first 2 urgent items
    if reviewed == 2:
        break

print("Business Note:")
print("This automation helps managers quickly identify low-stock items,")
print("prioritize urgent shortages, and prepare reorder actions faster.")

===== Inventory Summary =====
Total low-stock items: 4
Urgent low-stock items: 3
Low-stock Item IDs: ['A101', 'B201', 'C301', 'C302']
Urgent Item IDs: ['A101', 'B201', 'C301']

===== Reorder Report =====
  Item_ID        Item_Name         Category  Current_Stock  Reorder_Level  \
0    A101    Printer Paper  Office Supplies             12             20   
1    B201      Coffee Pods         Beverage              8             15   
2    C301  Shipping Labels        Warehouse              4             10   
3    C302     Packing Tape        Warehouse             18             18   

   Shortage      Supplier  
0         8    Staples Co  
1         7  Coffee World  
2         6       PackPro  
3         0       PackPro  

===== Urgent Reorder Preview =====
Review Item: A101 - Printer Paper
Supplier: Staples Co
Shortage: 8

Review Item: B201 - Coffee Pods
Supplier: Coffee World
Shortage: 7

Business Note:
This automation helps managers quickly identify low-stock items,
prioritize urgent 

# Homework 4 - Problem 10
## Tracking Task Deadlines and Follow-Ups

This notebook reviews task records, identifies overdue and upcoming work,
and creates a reminder report to support team coordination.

In [ ]:
import pandas as pd

# Sample task data
tasks = [
    {"Task_ID": "T001", "Task_Name": "Update shipment log", "Assigned_To": "Ryan",
     "Due_Date": "2026-04-10", "Priority": "High", "Status": "Open"},

    {"Task_ID": "T002", "Task_Name": "Review delayed orders", "Assigned_To": "Maria",
     "Due_Date": "2026-04-14", "Priority": "High", "Status": "In Progress"},

    {"Task_ID": "T003", "Task_Name": "Send vendor follow-up", "Assigned_To": "John",
     "Due_Date": "2026-04-18", "Priority": "Medium", "Status": "Open"},

    {"Task_ID": "T004", "Task_Name": "Count damaged inventory", "Assigned_To": "Ryan",
     "Due_Date": "2026-04-09", "Priority": "High", "Status": "Completed"},

    {"Task_ID": "T005", "Task_Name": "Check missing documents", "Assigned_To": "Nina",
     "Due_Date": "2026-04-13", "Priority": "Low", "Status": "Open"},

    {"Task_ID": "T006", "Task_Name": "Prepare weekly report", "Assigned_To": "Ryan",
     "Due_Date": "2026-04-20", "Priority": "Medium", "Status": "Open"}
]

# Convert list of dictionaries to DataFrame
df = pd.DataFrame(tasks)

# Convert Due_Date to datetime
df["Due_Date"] = pd.to_datetime(df["Due_Date"])

# Set today's date for comparison
today = pd.Timestamp("2026-04-12")

# Lists to store flagged tasks
overdue_tasks = []
upcoming_tasks = []
reminder_rows = []

# Review each task
for i in range(len(df)):
    row = df.loc[i]

    # Skip completed tasks because they do not need reminders
    if row["Status"] == "Completed":
        continue

    days_until_due = (row["Due_Date"] - today).days

    # Overdue tasks
    if days_until_due < 0:
        overdue_tasks.append(row["Task_ID"])

    # Upcoming tasks due within the next 7 days
    elif days_until_due <= 7:
        upcoming_tasks.append(row["Task_ID"])

    # Save flagged tasks into reminder list
    if row["Task_ID"] in overdue_tasks or row["Task_ID"] in upcoming_tasks:
        reminder_rows.append({
            "Task_ID": row["Task_ID"],
            "Task_Name": row["Task_Name"],
            "Assigned_To": row["Assigned_To"],
            "Due_Date": row["Due_Date"],
            "Priority": row["Priority"],
            "Status": row["Status"],
            "Days_Until_Due": days_until_due
        })

# Convert to DataFrame
reminder_report = pd.DataFrame(reminder_rows)

# Sort by priority and due date
priority_order = {"High": 1, "Medium": 2, "Low": 3}
reminder_report["Priority_Sort"] = reminder_report["Priority"].map(priority_order)
reminder_report = reminder_report.sort_values(by=["Priority_Sort", "Due_Date"])
reminder_report = reminder_report.drop(columns=["Priority_Sort"])

print("===== Task Reminder Summary =====")
print("Overdue tasks:", len(overdue_tasks))
print("Upcoming tasks in next 7 days:", len(upcoming_tasks))
print("Overdue Task IDs:", overdue_tasks)
print("Upcoming Task IDs:", upcoming_tasks)
print()

print("===== Final Reminder Report =====")
print(reminder_report)
print()

# While loop example: review the most urgent reminders
print("===== Priority Review Preview =====")
position = 0
reviewed = 0

while position < len(reminder_report):
    current_task = reminder_report.iloc[position]

    # Skip low-priority tasks during urgent preview
    if current_task["Priority"] == "Low":
        position += 1
        continue

    print("Review Task:", current_task["Task_ID"], "-", current_task["Task_Name"])
    print("Assigned To:", current_task["Assigned_To"])
    print("Priority:", current_task["Priority"])
    print("Days Until Due:", current_task["Days_Until_Due"])
    print()

    reviewed += 1
    position += 1

    # Stop after reviewing first 3 urgent tasks
    if reviewed == 3:
        break

print("Business Note:")
print("This automation helps managers quickly identify overdue work,")
print("upcoming deadlines, and the most urgent follow-up tasks.")

# Homework 4 - Problem 7
## Organizing Customer or Client Email Responses

This notebook generates personalized email response drafts based on request type,
account status, and reference dates.

In [ ]:
import pandas as pd

# Sample customer message data
requests = [
    {"Customer_Name": "Alice Brown", "Email": "alice@example.com",
     "Request_Type": "Appointment Confirmation", "Account_Status": "Active",
     "Reference_Date": "2026-04-18"},

    {"Customer_Name": "Brian Lee", "Email": "brian@example.com",
     "Request_Type": "Payment Reminder", "Account_Status": "Past Due",
     "Reference_Date": "2026-04-15"},

    {"Customer_Name": "Carla Gomez", "Email": "carla@example.com",
     "Request_Type": "Shipping Update", "Account_Status": "Active",
     "Reference_Date": "2026-04-20"},

    {"Customer_Name": "David King", "Email": "david@example.com",
     "Request_Type": "Payment Reminder", "Account_Status": "Active",
     "Reference_Date": "2026-04-16"},

    {"Customer_Name": "Emma Scott", "Email": "emma@example.com",
     "Request_Type": "Appointment Confirmation", "Account_Status": "Active",
     "Reference_Date": "2026-04-19"}
]

# Convert to DataFrame
df = pd.DataFrame(requests)

# Convert date column
df["Reference_Date"] = pd.to_datetime(df["Reference_Date"])

# List to store final email drafts
email_drafts = []

# Generate message drafts
for i in range(len(df)):
    row = df.loc[i]

    # Skip records with missing email
    if row["Email"] == "":
        continue

    # Template 1: Appointment confirmation
    if row["Request_Type"] == "Appointment Confirmation":
        message = (
            f"Hello {row['Customer_Name']},\\n\\n"
            f"This is a confirmation that your appointment is scheduled for "
            f"{row['Reference_Date'].date()}. Please contact us if you need to reschedule.\\n\\n"
            f"Best regards,\\nCustomer Support"
        )

    # Template 2: Payment reminder
    elif row["Request_Type"] == "Payment Reminder":
        if row["Account_Status"] == "Past Due":
            message = (
                f"Hello {row['Customer_Name']},\\n\\n"
                f"Our records show that your account is past due. Please submit payment by "
                f"{row['Reference_Date'].date()} to avoid any service interruption.\\n\\n"
                f"Best regards,\\nBilling Team"
            )
        else:
            message = (
                f"Hello {row['Customer_Name']},\\n\\n"
                f"This is a friendly reminder that your payment is due on "
                f"{row['Reference_Date'].date()}.\\n\\n"
                f"Best regards,\\nBilling Team"
            )

    # Template 3: Shipping update
    elif row["Request_Type"] == "Shipping Update":
        message = (
            f"Hello {row['Customer_Name']},\\n\\n"
            f"Your shipment is currently being processed. The expected update date is "
            f"{row['Reference_Date'].date()}. We will notify you if anything changes.\\n\\n"
            f"Best regards,\\nLogistics Team"
        )

    else:
        continue

    email_drafts.append({
        "Customer_Name": row["Customer_Name"],
        "Email": row["Email"],
        "Request_Type": row["Request_Type"],
        "Draft_Message": message
    })

# Convert drafts into DataFrame
draft_report = pd.DataFrame(email_drafts)

print("===== Email Draft Summary =====")
print("Total drafted messages:", len(draft_report))
print()

print("===== Final Draft Table =====")
print(draft_report[["Customer_Name", "Email", "Request_Type"]])
print()

# While loop example: preview draft messages one by one
print("===== Draft Preview =====")
position = 0
preview_count = 0

while position < len(draft_report):
    current_email = draft_report.iloc[position]

    # Skip preview of shipping update if needed
    if current_email["Request_Type"] == "Shipping Update":
        position += 1
        continue

    print("To:", current_email["Email"])
    print("Type:", current_email["Request_Type"])
    print(current_email["Draft_Message"])
    print("--------------------------------------------------")

    preview_count += 1
    position += 1

    # Stop after previewing first 3 messages
    if preview_count == 3:
        break

print("Business Note:")
print("This automation helps save time by generating consistent email drafts")
print("for common customer requests such as reminders, confirmations, and updates.")

**Custom Problem 5 — Escalating High-Priority Orders at Risk of Missing Shipment Deadlines**

My custom problem is escalating high-priority orders at risk of missing shipment deadlines. In supply chain work, urgent customer orders need faster attention when they are close to the promised ship date or already overdue. This problem would review order records, flag urgent and at-risk orders, and generate an escalation summary for management. I chose this problem because I work in supply chain and my role involves helping a team process orders and make shipments on time.

## AI Use Statement

I used AI tools such as ChatGPT to help me understand the business problems,
organize my ideas, and refine my Python code. I reviewed the final notebook
myself, ran the code on my own, and I am responsible for the final submission
and video walkthrough.